# 02 — Your First Neural Network (MNIST)

**INSEA Innovation Edge · Deep Learning Workshop**

---

In notebook 1 we computed weighted sums, losses, and gradients **by hand with NumPy**.
Now we let **PyTorch** do all that work for us — and train a real network on
the classic **MNIST** dataset of handwritten digits (0–9).

**What you'll see:**
- How concepts from notebook 1 map to PyTorch lines of code
- A complete training loop: forward → loss → backward → update
- What happens when you break things (bad learning rate!)

> **The mapping from notebook 1:**
> - `np.dot(...)` → `nn.Linear(...)`
> - manual gradient math → `loss.backward()`
> - `w = w - lr * grad` → `optimizer.step()`

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

# Reproducibility
torch.manual_seed(0)

# Use GPU if available, else CPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## Section 1 — Load the Data

MNIST = 70,000 grayscale images of handwritten digits, each 28×28 pixels.
- 60,000 images for **training**
- 10,000 images for **testing**

`torchvision` downloads and formats it for us. `ToTensor()` converts each image
to a PyTorch tensor with pixel values scaled to the `[0, 1]` range.

In [ ]:
transform = transforms.ToTensor()

train_data = datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
test_data  = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# DataLoader = feeds the model small batches at a time
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_data,  batch_size=64, shuffle=False)

print(f'Training images: {len(train_data)}')
print(f'Test images:     {len(test_data)}')
print(f'Image shape:     {train_data[0][0].shape}  (channels, height, width)')

### Let's see a few images

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(10, 2))
for i, ax in enumerate(axes):
    image, label = train_data[i]
    ax.imshow(image.squeeze(), cmap='gray')
    ax.set_title(f'Label: {label}')
    ax.axis('off')
plt.tight_layout()
plt.show()

## Section 2 — Build the Network

Our model is a **Multi-Layer Perceptron (MLP)** with 2 hidden layers.

- **Input:** 784 numbers (28 × 28 pixels flattened)
- **Hidden layer 1:** 128 neurons + ReLU
- **Hidden layer 2:** 64 neurons + ReLU
- **Output:** 10 numbers (one score per digit class)

Each `nn.Linear(a, b)` is exactly the dot product from notebook 1 —
just many of them in parallel, with weights PyTorch learns for us.

In [ ]:
class DigitClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28 * 28, 128)   # input  -> hidden 1
        self.fc2 = nn.Linear(128, 64)         # hidden 1 -> hidden 2
        self.fc3 = nn.Linear(64, 10)          # hidden 2 -> output (10 classes)

    def forward(self, x):
        x = x.view(x.size(0), -1)             # flatten 28x28 into 784
        x = F.relu(self.fc1(x))               # dot product + ReLU
        x = F.relu(self.fc2(x))               # dot product + ReLU
        x = self.fc3(x)                       # final scores (logits)
        return x

model = DigitClassifier().to(device)
print(model)

## Section 3 — Pick a Loss and an Optimizer

- **Loss function:** `CrossEntropyLoss` — the standard choice for multi-class
  classification. It combines Softmax (from notebook 1) + a log-likelihood loss.
- **Optimizer:** `SGD` with a learning rate — exactly the weight-update rule
  from notebook 1: `new_weight = old_weight - lr * gradient`.

In [ ]:
loss_fn   = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

## Section 4 — Train the Network

The training loop does the same four steps for every batch:

1. **Forward pass** — predict
2. **Compute loss** — how wrong?
3. **Backward pass** — compute gradients (`loss.backward()`)
4. **Update weights** — take a step downhill (`optimizer.step()`)

We'll train for **3 epochs** (one epoch = one full pass over the training set).

In [ ]:
EPOCHS = 3
train_losses = []

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        # 1. Forward
        predictions = model(images)

        # 2. Loss
        loss = loss_fn(predictions, labels)

        # 3. Backward  (computes all gradients)
        optimizer.zero_grad()
        loss.backward()

        # 4. Update weights
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    train_losses.append(avg_loss)
    print(f'Epoch {epoch+1}/{EPOCHS}  —  avg loss: {avg_loss:.4f}')

In [ ]:
plt.figure(figsize=(7, 3))
plt.plot(range(1, EPOCHS + 1), train_losses, marker='o', color='#1E6FCC')
plt.xlabel('Epoch')
plt.ylabel('Training loss')
plt.title('Loss goes down — the network is learning')
plt.grid(alpha=0.3)
plt.show()

## Section 5 — Evaluate on the Test Set

Training loss tells us the network memorized the training data.
The real question: does it work on **unseen** images?

In [ ]:
model.eval()
correct = 0
total   = 0

with torch.no_grad():                         # no gradients during evaluation
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        predictions = model(images)
        predicted_labels = predictions.argmax(dim=1)
        correct += (predicted_labels == labels).sum().item()
        total   += labels.size(0)

accuracy = 100 * correct / total
print(f'Test accuracy: {accuracy:.2f}%  ({correct}/{total})')

### Let's look at some predictions

In [ ]:
model.eval()
images, labels = next(iter(test_loader))
images, labels = images.to(device), labels.to(device)

with torch.no_grad():
    preds = model(images).argmax(dim=1)

fig, axes = plt.subplots(2, 6, figsize=(12, 4))
for i, ax in enumerate(axes.flat):
    img = images[i].cpu().squeeze()
    true = labels[i].item()
    pred = preds[i].item()
    color = 'green' if pred == true else 'red'
    ax.imshow(img, cmap='gray')
    ax.set_title(f'Pred: {pred} | True: {true}', color=color, fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.show()

## Section 6 — Break It On Purpose (Bad Learning Rate)

In notebook 1 we saw what happens when the learning rate is too large:
**the loss explodes**. Let's reproduce that on a real network.

We'll rebuild a fresh model and train it with `lr = 10` (100× too large).

In [ ]:
bad_model     = DigitClassifier().to(device)
bad_optimizer = torch.optim.SGD(bad_model.parameters(), lr=10.0)   # way too big!

bad_losses = []
bad_model.train()

# Just a few batches is enough to see divergence
for i, (images, labels) in enumerate(train_loader):
    if i >= 30:
        break
    images, labels = images.to(device), labels.to(device)
    preds = bad_model(images)
    loss  = loss_fn(preds, labels)
    bad_optimizer.zero_grad()
    loss.backward()
    bad_optimizer.step()
    bad_losses.append(loss.item())

plt.figure(figsize=(7, 3))
plt.plot(bad_losses, color='red', linewidth=2)
plt.xlabel('Batch')
plt.ylabel('Loss')
plt.title('lr = 10.0  —  loss diverges instead of going down')
plt.grid(alpha=0.3)
plt.show()

print(f'Final loss: {bad_losses[-1]:.2f}  (should be ~0.1 if training worked)')

## Recap

- **`nn.Linear`** is the dot product from notebook 1
- **`loss.backward()`** runs backpropagation — fills every parameter's gradient
- **`optimizer.step()`** applies the weight update using those gradients
- **Learning rate matters** — too big explodes, too small crawls

You just built, trained, and evaluated a neural network from scratch.

## What's Next

Open **`03_finetune_workshop.ipynb`** — instead of training from zero,
you'll take a pre-trained language model and adapt it to a new task
in a fraction of the time.

---

*INSEA Innovation Edge · Deep Learning Workshop*